In [0]:
%sql
-- Distinct values + counts for order_status
SELECT order_status, COUNT(*) 
FROM ecommerce.base.orders 
GROUP BY order_status 
ORDER BY COUNT(*) DESC;

In [0]:
%sql
-- Distinct payment_method values (dictionary flags inconsistent spelling)
SELECT payment_method, COUNT(*) 
FROM ecommerce.base.payments 
GROUP BY payment_method 
ORDER BY COUNT(*) DESC;

In [0]:
%sql
-- Distinct city values in customers (dictionary flags inconsistency)
SELECT city, COUNT(*) 
FROM ecommerce.base.customers 
GROUP BY city 
ORDER BY COUNT(*) DESC;

In [0]:
%sql
SELECT
    LOWER(TRIM(city)) AS standardized_city,
    COUNT(DISTINCT city) AS variations,
    COUNT(*) AS total_rows
FROM ecommerce.base.customers
WHERE city IS NOT NULL
GROUP BY LOWER(TRIM(city))
HAVING COUNT(DISTINCT city) > 1
ORDER BY variations DESC;

In [0]:
%sql
-- Invalid birth_date check (future dates, absurdly old dates)
SELECT customer_id, birth_date
FROM ecommerce.base.customers
WHERE birth_date > CURRENT_DATE
   OR birth_date < DATE '1900-01-01';

In [0]:
%sql
-- Returns table profiling: nulls, return_reason distribution, refund_amount range
SELECT
  COUNT(*) AS total_returns,
  COUNT(*) - COUNT(return_reason) AS null_reason,
  COUNT(*) - COUNT(refund_amount) AS null_refund,
  MIN(refund_amount) AS min_refund,
  MAX(refund_amount) AS max_refund,
  AVG(refund_amount) AS avg_refund
FROM ecommerce.base.returns;

In [0]:
%sql
-- Distinct return_reason values and their frequency
SELECT return_reason, COUNT(*) AS occurrences
FROM ecommerce.base.returns
GROUP BY return_reason
ORDER BY occurrences DESC;

In [0]:
%sql
-- Payment status distribution + mismatch check against orders.order_total
SELECT
  p.payment_status,
  COUNT(*) AS occurrences,
  SUM(CASE WHEN p.amount != o.order_total THEN 1 ELSE 0 END) AS amount_mismatch_count
FROM ecommerce.base.payments p
JOIN ecommerce.base.orders o ON p.order_id = o.order_id
GROUP BY p.payment_status
ORDER BY occurrences DESC;